### PHASE 1: GENERATING THE "STABILITY & AD-HEALTH" LOGS


In [1]:
import pandas as pd
import numpy as np

def generate_sentinel_logs(samples=1000000):
    np.random.seed(42)
    print(f"Generating {samples} logs... analyzing stability and ad-health.")
    
    data = {
        'session_id': range(samples),
        'device_ram_gb': np.random.choice([2, 4, 6, 8, 12], samples, p=[0.2, 0.3, 0.2, 0.2, 0.1]),
        'fps_avg': np.random.normal(58, 5, samples),
        'ads_requested': np.random.randint(1, 20, samples),
        'level_id': np.random.randint(1, 150, samples),
        'session_duration_min': np.random.uniform(1, 30, samples)
    }
    
    df = pd.DataFrame(data)
    
    # Logic for 'Ads Shown' (Normal = 90% fill rate)
    df['ads_shown'] = (df['ads_requested'] * np.random.uniform(0.8, 1.0, samples)).astype(int)
    
    # Initialize Flags
    df['crash_flag'] = 0
    df['anr_flag'] = 0

    # INJECTING THE ANOMALIES (The stuff the Agent must find)
    
    # 1. THE AD LEAK: (0.5% of cases) Requested ads but showed 0
    ad_leak_idx = np.random.choice(df.index, 5000, replace=False)
    df.loc[ad_leak_idx, 'ads_shown'] = 0
    
    # 2. THE CRASH: Low RAM devices on high levels (0.3% of cases)
    crash_idx = df[(df['device_ram_gb'] <= 2) & (df['level_id'] > 100)].sample(frac=0.1).index
    df.loc[crash_idx, 'crash_flag'] = 1
    df.loc[crash_idx, 'fps_avg'] = np.random.uniform(0, 5, len(crash_idx))
    
    # 3. THE ANR (App Not Responding): Low FPS + High Session duration
    anr_idx = df[df['fps_avg'] < 20].sample(frac=0.2).index
    df.loc[anr_idx, 'anr_flag'] = 1

    return df

df_p3 = generate_sentinel_logs()
df_p3.to_csv('p3_sentinel_logs.csv', index=False)

print("\n--- SENTINEL DATA AUDIT ---")
print(f"Total Logs: {len(df_p3)}")
print(f"Crashes Found: {df_p3['crash_flag'].sum()}")
print(f"ANRs Found: {df_p3['anr_flag'].sum()}")
print(f"Ad-Leaks (Zero Ads): {len(df_p3[df_p3['ads_shown'] == 0])}")

Generating 1000000 logs... analyzing stability and ad-health.

--- SENTINEL DATA AUDIT ---
Total Logs: 1000000
Crashes Found: 6553
ANRs Found: 1311
Ad-Leaks (Zero Ads): 57140


### PHASE 2: BUILDING THE "SENTINEL" (ISOLATION FOREST)

In [4]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 1. LOAD DATA
df_p3 = pd.read_csv('p3_sentinel_logs.csv')

# 2. SELECT FEATURES
# These are the 'Vital Signs' of the game.
features = ['device_ram_gb', 'fps_avg', 'ads_requested', 'ads_shown', 'level_id']
X = df_p3[features]

# 3. SCALE 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. THE ISOLATION FOREST (The Agent)
# Corrected: No space in 'IsolationForest'
# n_jobs=-1 uses all your CPU cores to make it faster
iso_forest = IsolationForest(contamination=0.07, random_state=42, n_jobs=-1)

print("Sentinel Agent is scanning 1,000,000 logs for anomalies... please wait.")
# -1 = Anomaly, 1 = Normal
df_p3['anomaly_score'] = iso_forest.fit_predict(X_scaled)

anomalies_detected = (df_p3['anomaly_score'] == -1).sum()

print(f"\nPHASE 2 COMPLETE.")
print(f"Total Anomalies Flagged by Agent: {anomalies_detected}")

Sentinel Agent is scanning 1,000,000 logs for anomalies... please wait.

PHASE 2 COMPLETE.
Total Anomalies Flagged by Agent: 70000


### PHASE 3: THE DIAGNOSTIC (Investigating the 70,000)

In [5]:
# 1. Split the data into Normal and Anomaly groups
normal_sessions = df_p3[df_p3['anomaly_score'] == 1]
anomalous_sessions = df_p3[df_p3['anomaly_score'] == -1]

# 2. Compare the averages
print("--- THE SENTINEL DIAGNOSTIC REPORT ---")
print("\n[NORMAL SESSIONS] Average Stats:")
print(normal_sessions[['fps_avg', 'ads_shown', 'device_ram_gb']].mean())

print("\n[ANOMALY SESSIONS] Average Stats:")
print(anomalous_sessions[['fps_avg', 'ads_shown', 'device_ram_gb']].mean())

# 3. THE REVENUE LEAK DETECTOR
leaks = anomalous_sessions[anomalous_sessions['ads_shown'] == 0]
print(f"\n[REVENUE ALERT]: The Agent found {len(leaks)} sessions where zero ads were shown.")

# 4. THE CRASH DETECTOR
crashes = anomalous_sessions[anomalous_sessions['fps_avg'] < 10]
print(f"[TECHNICAL ALERT]: The Agent found {len(crashes)} sessions with 'Near-Zero' FPS (Possible Crashes).")

--- THE SENTINEL DIAGNOSTIC REPORT ---

[NORMAL SESSIONS] Average Stats:
fps_avg          57.897817
ads_shown         8.362531
device_ram_gb     5.449187
dtype: float64

[ANOMALY SESSIONS] Average Stats:
fps_avg          54.031308
ads_shown         8.638957
device_ram_gb     7.640371
dtype: float64

[REVENUE ALERT]: The Agent found 16395 sessions where zero ads were shown.
[TECHNICAL ALERT]: The Agent found 6553 sessions with 'Near-Zero' FPS (Possible Crashes).
